In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
import sys
sys.path.append('..')
from langchain_chat.config import get_deepseek_key

# 提示词模板用来组织对话消息的结构
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Translate the following from chinese into {language}"),
    ("user", "{text}")
])

# 构建 DeepSeek 模型客户端
llm = ChatOpenAI(
    model="deepseek-chat",
    base_url="https://api.deepseek.com/v1",
    openai_api_key=get_deepseek_key(),
)

# 结果解析器
parser = StrOutputParser()

# 构建链
chain = prompt_template | llm | parser

# 调用链
result = chain.invoke({"text": "我草泥马", "language": "汉语"})
print(result)

api_key:sk-44e6f5c6fde84d53b43e87c07e82e2b3
对不起，我还没有学会回答这个问题。如果你有其他问题，我非常乐意为你提供帮助。


在LangChain的LCEL（LangChain Expression Language）中，| 管道操作符要求两边的对象都实现Runnable接口，而Runnable接口的核心就是invoke方法。

In [8]:


analysis_prompt = ChatPromptTemplate.from_template("我应该怎么回答这句话? {talk} 。给我一个五个字以内的示例")
chain2 = talk=chain | analysis_prompt | llm | parser
print(chain2.invoke({"text": "nice to meet you", "language": "Chinese"}))

示例：  
“幸会。”  
“同喜。”


构建多个并⾏的链，构建更复杂的⼤模型应⽤逻辑，通过RunnableMap去并行执行这两个链

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableMap, RunnableLambda, RunnableWithMessageHistory
# 提示词模板
prompt_template_zh = ChatPromptTemplate.from_messages([
  ("system", "Translate the following from English into Chinese"),
  ("user", "{text}")
])
prompt_template_fr = ChatPromptTemplate.from_messages([
  ("system", "Translate the following from English into French"),
  ("user", "{text}")
])
prompt_template_tk = ChatPromptTemplate.from_messages([
  ("system", "Translate the following from English into 日语"),
  ("user", "{text}")
])
# 构建链
chain_zh = prompt_template_zh | llm | parser
chain_fr = prompt_template_fr | llm | parser
chain_tk = prompt_template_tk | llm | parser
print(chain_fr)
print("-------")
# 并⾏执⾏两个链
parallel_chains = RunnableMap({
"zh_translation": chain_zh,
"fr_translation": chain_fr,
"tk_translation": chain_tk
})
prompt_template_fr = ChatPromptTemplate.from_messages([
  ("system", "Translate the following from English into French"),
  ("user", "{text}")
])
# 合并结果
final_chain = parallel_chains | RunnableLambda(
    lambda x: f"Chinese: {x['zh_translation']}\nFrench: {x['fr_translation']}\n日语: {x['tk_translation']}"
)
# 调⽤链
print(final_chain.invoke({"text": "nice to meet you"}))


first=ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Translate the following from English into French'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})]) middle=[ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001ABFF5BF450>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001ABFF5BE550>, root_client=<openai.OpenAI object at 0x000001ABFF587FD0>, root_async_client=<openai.AsyncOpenAI object at 0x000001ABFF5BE5D0>, model_name='deepseek-chat', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.deepseek.com/v1')] last=StrOutputParser()
-------
Chinese: 很高兴见到你
French: enchanté de 

In [ ]:
!pip install langgraph
!pip install grandalf

In [ ]:
# 实际上，LangChain提供了⼀个顶级⽗类，Runnable。只要是Runnable的⼦类，就都可以链接成⼀个序列。
final_chain.get_graph().print_ascii()

            +-------------------------------------------------------------+            
            | Parallel<zh_translation,fr_translation,tk_translation>Input |            
            +-------------------------------------------------------------+            
                         *****             *             *****                         
                   ******                  *                  ******                   
                ***                        *                        ***                
+--------------------+          +--------------------+          +--------------------+ 
| ChatPromptTemplate |          | ChatPromptTemplate |          | ChatPromptTemplate | 
+--------------------+          +--------------------+          +--------------------+ 
           *                               *                               *           
           *                               *                               *           
           *                    

In [27]:
from langchain_core.chat_history import InMemoryChatMessageHistory
history = InMemoryChatMessageHistory()
print("第⼀轮聊天")
history.add_user_message("现在java常用的web框架是什么")
aimessage = llm.invoke(history.messages)
print(aimessage)
history.add_ai_message(aimessage)
history.clear()
print("第⼆轮聊天")
history.add_user_message("请重复⼀次")
aimessage = llm.invoke(history.messages)
print(aimessage.content)


第⼀轮聊天
content='目前Java Web开发中常用的框架主要分为以下几类：\n\n## 一、主流Web框架\n\n### 1. **Spring Boot**（最主流）\n- **特点**：约定优于配置，快速搭建项目\n- **生态**：Spring全家桶（Spring MVC, Spring Security, Spring Data等）\n- **适用场景**：企业级应用、微服务、REST API\n\n### 2. **Spring MVC**\n- **特点**：传统的MVC框架，Spring Boot内置\n- **适用场景**：传统Web应用\n\n### 3. **Micronaut**\n- **特点**：轻量级，启动快，内存占用小\n- **适用场景**：微服务、Serverless、云原生\n\n### 4. **Quarkus**\n- **特点**：云原生，GraalVM原生镜像支持\n- **适用场景**：Kubernetes环境、Serverless\n\n### 5. **Vert.x**\n- **特点**：响应式、事件驱动\n- **适用场景**：高并发、实时应用\n\n## 二、微服务框架\n\n### 1. **Spring Cloud**（最流行）\n- 基于Spring Boot的微服务全家桶\n- 包含服务发现、配置中心、网关等组件\n\n### 2. **Dubbo**\n- **特点**：阿里巴巴开源的高性能RPC框架\n- **适用场景**：大型分布式系统\n\n## 三、模板引擎\n\n### 1. **Thymeleaf**（Spring官方推荐）\n- 自然模板，可直接在浏览器中预览\n\n### 2. **FreeMarker**\n- 老牌模板引擎，功能强大\n\n### 3. **JSP**（逐渐淘汰）\n- 传统Java Web技术，现在使用较少\n\n## 四、其他重要框架\n\n### 1. **MyBatis / MyBatis-Plus**\n- 持久层框架，简化数据库操作\n\n### 2. **Hibernate**\n- JPA实现，对象关系映射\n\n### 3. **JUnit 5 / TestNG**\n- 测试框架\n\n### 4. **Lombok**\n- 减少样板

In [28]:
!pip install -q langchain-redis redis